# PUMA Colab Training (Project Scripts + W&B)

Notebook nay tao moi cho Colab.

Flow:
1. Clone repo + cai dependencies.
2. Mount Google Drive (dataset + checkpoint).
3. Convert dataset_PUMA ve format train cua project.
4. Tao config train dung input size 1024 va dung so class theo track.
5. Chay `scripts/train.py` va theo doi bang Weights & Biases.

In [ ]:
# ===== 1) Clone project repo =====
# Ban co the doi REPO_URL theo fork cua ban.
REPO_URL = 'https://github.com/hoangtung386/PUMA_Nucleus_Segmentation.git'
REPO_DIR = 'PUMA_Nucleus_Segmentation'

!rm -rf {REPO_DIR}
!git clone {REPO_URL}
%cd {REPO_DIR}
!ls -la

In [ ]:
# ===== 2) Install dependencies =====
!pip -q install -U pip
!pip -q install -e .[dev,cellpose]
!pip -q install wandb

In [ ]:
# ===== 3) Mount Drive and define paths =====
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content') / REPO_DIR

# Dataset root trong Drive (giu nguyen cau truc dataset_PUMA)
DATASET_ROOT = Path('/content/drive/MyDrive/dataset_PUMA')

RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
SPLIT_DIR = REPO_ROOT / 'data' / 'splits'

# Noi luu train outputs/checkpoint tren Drive
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/PUMA_experiments')
CKPT_ROOT = DRIVE_RUN_ROOT / 'models'
LOG_ROOT = DRIVE_RUN_ROOT / 'runs'
OUT_ROOT = DRIVE_RUN_ROOT / 'outputs'

for p in [RAW_DIR / 'images', RAW_DIR / 'annotations', PROCESSED_DIR, SPLIT_DIR, CKPT_ROOT, LOG_ROOT, OUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('CKPT_ROOT:', CKPT_ROOT)

In [ ]:
# ===== 4) Select track and sanity checks =====
TRACK = 1  # 1 or 2
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10
SEED = 42

# Epochs: can tune based on GPU budget
SEG_EPOCHS = 20
CLS_PHASE1_EPOCHS = 5
CLS_PHASE2_EPOCHS = 15

from puma_seg.data.geojson_parser import NUM_CLASSES
n_fg = NUM_CLASSES[TRACK] - 1
assert (TRACK == 1 and n_fg == 3) or (TRACK == 2 and n_fg == 10)
print(f'Track {TRACK} -> foreground classes: {n_fg}')
print('Input size set later as image_size=None (keep 1024x1024).')

In [ ]:
# ===== 5) Build raw dataset expected by scripts/prepare_data.py =====
# Expected layout:
#   data/raw/images/<stem>.tif
#   data/raw/annotations/<stem>.geojson
# Source nuclei anno in dataset_PUMA: <stem>_nuclei.geojson

import shutil

img_src_dir = DATASET_ROOT / '01_training_dataset_tif_ROIs'
ann_src_dir = DATASET_ROOT / '01_training_dataset_geojson_nuclei'
assert img_src_dir.exists(), f'Missing {img_src_dir}'
assert ann_src_dir.exists(), f'Missing {ann_src_dir}'

img_dst_dir = RAW_DIR / 'images'
ann_dst_dir = RAW_DIR / 'annotations'

for p in img_dst_dir.glob('*'):
    p.unlink()
for p in ann_dst_dir.glob('*'):
    p.unlink()

copied, missing = 0, []
for img_path in sorted(img_src_dir.glob('*.tif')):
    stem = img_path.stem
    ann_path = ann_src_dir / f'{stem}_nuclei.geojson'
    if not ann_path.exists():
        missing.append(stem)
        continue
    shutil.copy2(img_path, img_dst_dir / f'{stem}.tif')
    shutil.copy2(ann_path, ann_dst_dir / f'{stem}.geojson')
    copied += 1

print('Copied pairs:', copied)
print('Missing annotations:', len(missing))
print('Images:', len(list(img_dst_dir.glob('*.tif'))))
print('Annotations:', len(list(ann_dst_dir.glob('*.geojson'))))

In [ ]:
# ===== 6) Prepare processed data =====
import subprocess
import sys

prepare_cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'prepare_data.py'),
    '--raw-dir', str(RAW_DIR),
    '--out-dir', str(PROCESSED_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--track', str(TRACK),
    '--val-split', str(VAL_SPLIT),
    '--test-split', str(TEST_SPLIT),
    '--seed', str(SEED),
    '--image-ext', '.tif',
]
print('Running:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=REPO_ROOT, check=True)

In [ ]:
# ===== 7) Login W&B and init run =====
import os
import yaml
from datetime import datetime
import wandb

# Cach 1: interactive login
wandb.login()

run_name = f'track{TRACK}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

cfg = {
    'data': {
        'raw_dir': str(RAW_DIR),
        'processed_dir': str(PROCESSED_DIR),
        'split_file': str(SPLIT_DIR / 'split.json'),
        'track': TRACK,
        'image_size': None,  # keep original 1024x1024
        'val_split': VAL_SPLIT,
        'test_split': TEST_SPLIT,
        'crop_size': 64,
    },
    'segmentation': {
        'pretrained_model': 'cyto3',
        'gpu': True,
        'diameter': 17.0,
        'nchan': 2,
        'channels': [0, 0],
        'learning_rate': 1e-5,
        'weight_decay': 0.1,
        'n_epochs': SEG_EPOCHS,
        'batch_size': 4,
        'model_name': f'puma_cellpose_{run_name}',
    },
    'classification': {
        'pretrained': True,
        'freeze_backbone': True,
        'dropout': 0.3,
        'phase1_epochs': CLS_PHASE1_EPOCHS,
        'phase1_lr': 1e-3,
        'phase2_epochs': CLS_PHASE2_EPOCHS,
        'phase2_lr_head': 1e-4,
        'phase2_lr_backbone': 1e-6,
        'unfreeze_groups': 2,
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'patience': 8,
        'batch_size': 32,
        'use_amp': True,
        'model_name': f'best_classifier_{run_name}',
    },
    'paths': {
        'save_dir': str(CKPT_ROOT / run_name),
        'log_dir': str(LOG_ROOT / run_name),
        'output_dir': str(OUT_ROOT / run_name),
    },
}

cfg_path = REPO_ROOT / 'configs' / f'colab_{run_name}.yaml'
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

wandb_run = wandb.init(
    project='puma-nucleus-segmentation',
    name=run_name,
    config=cfg,
    job_type='train',
)
wandb.save(str(cfg_path))
print('Config path:', cfg_path)

In [ ]:
# ===== 8) Train using project script =====
import subprocess
import sys

train_cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'train.py'),
    '--config', str(cfg_path),
    '--mode', 'both',
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
print('Training done.')

In [ ]:
# ===== 9) Log checkpoints as W&B artifact =====
from pathlib import Path

save_dir = Path(cfg['paths']['save_dir'])
log_dir = Path(cfg['paths']['log_dir'])

print('save_dir:', save_dir)
print('log_dir:', log_dir)

artifact = wandb.Artifact(name=f'puma-checkpoints-{run_name}', type='model')
if save_dir.exists():
    artifact.add_dir(str(save_dir))
if log_dir.exists():
    artifact.add_dir(str(log_dir))

wandb.log_artifact(artifact)
wandb.finish()

print('Uploaded artifact to W&B and finished run.')